# ETL — Incidentes Viales Medellín
**Proyecto ETL | ODS 3 — Salud y Bienestar**  
Pipeline completo: Extract → Transform → Load  
Primera Entrega

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, text

RAW_PATH = '../1_raw_data/incidentes_viales.csv'
DB_URL   = 'mysql+pymysql://root:@localhost:3306/dw_incidentes_v2'

print('Librerías cargadas correctamente')

---
## FASE 1 — EXTRACT

In [ ]:
print('=' * 50)
print('EXTRAYENDO datos del CSV...')

df_raw = pd.read_csv(
    RAW_PATH,
    encoding='utf-8',
    on_bad_lines='skip',
    low_memory=False
)

print(f'Filas extraidas:    {df_raw.shape[0]:,}')
print(f'Columnas extraidas: {df_raw.shape[1]}')
print('=' * 50)
df_raw.head()

In [ ]:
# Tipos de datos originales
df_raw.info()

---
## FASE 2 — TRANSFORM
### 2.1 Limpieza de datos

In [ ]:
df = df_raw.copy()

# 1. Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(df)}')

# 2. Limpiar columna AÑO (tiene valores con \r embebido)
df['AÑO'] = df['AÑO'].astype(str).str.strip().str.replace(r'\r', '', regex=True)
df['AÑO'] = pd.to_numeric(df['AÑO'], errors='coerce')
print(f'AÑO limpiado. Valores únicos: {sorted(df["AÑO"].dropna().astype(int).unique())}')

# 3. Parsear fechas
df['FECHA_ACCIDENTE'] = pd.to_datetime(df['FECHA_ACCIDENTE'], errors='coerce', dayfirst=True)
df['FECHA_SOLO'] = df['FECHA_ACCIDENTE'].dt.normalize()
print(f'Fechas parseadas. Nulos: {df["FECHA_ACCIDENTE"].isna().sum()}')

In [ ]:
# 4. Corregir encoding
df['GRAVEDAD_ACCIDENTE'] = df['GRAVEDAD_ACCIDENTE'].str.replace('Solo da\xf1os', 'Solo danos', regex=False)
df['DISEÑO'] = df['DISEÑO'].str.replace('Pont\xf3n', 'Ponton', regex=False)

# 5. Normalizar categorías inconsistentes
clase_map = {
    'Caida Ocupante'    : 'Caida de Ocupante',
    'Caída Ocupante'    : 'Caida de Ocupante',
    'Caida de Ocupante' : 'Caida de Ocupante',
}
df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].replace(clase_map)
print('Categorías normalizadas')
print(df['CLASE_ACCIDENTE'].value_counts().head())

In [ ]:
# 6. Imputar nulos
df['BARRIO']          = df['BARRIO'].fillna('DESCONOCIDO')
df['COMUNA']          = df['COMUNA'].fillna('DESCONOCIDO')
df['NUMCOMUNA']       = df['NUMCOMUNA'].fillna('DESCONOCIDO')
df['DISEÑO']          = df['DISEÑO'].fillna('DESCONOCIDO')
df['CLASE_ACCIDENTE'] = df['CLASE_ACCIDENTE'].fillna('Otro')
df['EXPEDIENTE']      = df['EXPEDIENTE'].fillna('SIN_EXPEDIENTE')
df['NRO_RADICADO']    = df['NRO_RADICADO'].fillna(0)

# 7. Eliminar columnas innecesarias
df = df.drop(columns=['FECHA_ACCIDENTES', 'CBML', 'LOCATION', 'DIRECCION ENCASILLADA'], errors='ignore')

print(f'Dataset limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Nulos restantes: {df.isnull().sum().sum()}')

### 2.2 Construcción del modelo dimensional

In [ ]:
# dim_tiempo
dim_tiempo = (
    df[['FECHA_SOLO', 'AÑO', 'MES']]
    .drop_duplicates()
    .dropna(subset=['FECHA_SOLO'])
    .reset_index(drop=True)
)
dim_tiempo.insert(0, 'sk_tiempo', range(1, len(dim_tiempo) + 1))
dim_tiempo.columns = ['sk_tiempo', 'fecha_accidente', 'anio', 'mes']
print(f'dim_tiempo:    {len(dim_tiempo):,} filas')
dim_tiempo.head()

In [ ]:
# dim_ubicacion
dim_ubicacion = (
    df[['BARRIO', 'COMUNA', 'NUMCOMUNA', 'DIRECCION', 'X', 'Y']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_ubicacion.insert(0, 'sk_ubicacion', range(1, len(dim_ubicacion) + 1))
dim_ubicacion.columns = ['sk_ubicacion', 'barrio', 'comuna', 'num_comuna', 'direccion', 'x', 'y']
print(f'dim_ubicacion: {len(dim_ubicacion):,} filas')
dim_ubicacion.head()

In [ ]:
# dim_accidente
dim_accidente = (
    df[['CLASE_ACCIDENTE', 'GRAVEDAD_ACCIDENTE', 'DISEÑO']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_accidente.insert(0, 'sk_accidente', range(1, len(dim_accidente) + 1))
dim_accidente.columns = ['sk_accidente', 'clase_accidente', 'gravedad_accidente', 'diseno_via']
print(f'dim_accidente: {len(dim_accidente):,} filas')
dim_accidente.head()

In [ ]:
# fact_incidente — JOIN con las dimensiones para obtener surrogate keys
df_fact = df.merge(
    dim_tiempo,
    left_on=['FECHA_SOLO', 'AÑO', 'MES'],
    right_on=['fecha_accidente', 'anio', 'mes'],
    how='left'
)
df_fact = df_fact.merge(
    dim_ubicacion,
    left_on=['BARRIO', 'COMUNA', 'NUMCOMUNA', 'DIRECCION', 'X', 'Y'],
    right_on=['barrio', 'comuna', 'num_comuna', 'direccion', 'x', 'y'],
    how='left'
)
df_fact = df_fact.merge(
    dim_accidente,
    left_on=['CLASE_ACCIDENTE', 'GRAVEDAD_ACCIDENTE', 'DISEÑO'],
    right_on=['clase_accidente', 'gravedad_accidente', 'diseno_via'],
    how='left'
)

fact_incidente = df_fact[['EXPEDIENTE', 'NRO_RADICADO', 'sk_tiempo', 'sk_ubicacion', 'sk_accidente']].copy()
fact_incidente.insert(0, 'sk_incidente', range(1, len(fact_incidente) + 1))
fact_incidente.columns = ['sk_incidente', 'expediente', 'nro_radicado', 'sk_tiempo', 'sk_ubicacion', 'sk_accidente']

print(f'fact_incidente: {len(fact_incidente):,} filas')
print(f'sk_tiempo nulos: {fact_incidente["sk_tiempo"].isna().sum()}')
fact_incidente.head()

### 2.3 Validación básica (pre-carga)

In [ ]:
errores = []

# Verificar nulos en columnas críticas
for col in ['sk_incidente', 'sk_ubicacion', 'sk_accidente', 'expediente']:
    nulos = fact_incidente[col].isna().sum()
    if nulos > 0:
        errores.append(f'CRITICO: {col} tiene {nulos} nulos')

# Verificar unicidad de sk_incidente
if fact_incidente['sk_incidente'].duplicated().sum() > 0:
    errores.append('CRITICO: sk_incidente tiene duplicados')

# Verificar rango de mes
fuera_rango = dim_tiempo[(dim_tiempo['mes'] < 1) | (dim_tiempo['mes'] > 12)]
if len(fuera_rango) > 0:
    errores.append(f'CRITICO: mes fuera de rango en {len(fuera_rango)} filas')

if errores:
    print('VALIDACION FALLIDA:')
    for e in errores: print(f'  {e}')
else:
    print('VALIDACION EXITOSA — Todos los checks criticos pasaron')
    print(f'  sk_tiempo nulos (no critico): {fact_incidente["sk_tiempo"].isna().sum()}')

---
## FASE 3 — LOAD

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(DB_URL)
print('Conectando al Data Warehouse...')

# Cargar dimensiones primero (integridad referencial)
dim_tiempo.to_sql('dim_tiempo', engine, if_exists='replace', index=False)
print(f'dim_tiempo cargada:     {len(dim_tiempo):,} filas')

dim_ubicacion.to_sql('dim_ubicacion', engine, if_exists='replace', index=False)
print(f'dim_ubicacion cargada:  {len(dim_ubicacion):,} filas')

dim_accidente.to_sql('dim_accidente', engine, if_exists='replace', index=False)
print(f'dim_accidente cargada:  {len(dim_accidente):,} filas')

# Cargar fact table en chunks
fact_incidente.to_sql('fact_incidente', engine, if_exists='replace', index=False, chunksize=5000)
print(f'fact_incidente cargada: {len(fact_incidente):,} filas')

print('\nCARGA COMPLETA al Data Warehouse dw_incidentes_v2')

---
## FASE 4 — VERIFICACIÓN DESDE EL DW
Todas las consultas se hacen desde el Data Warehouse, no desde el CSV.

In [ ]:
# KPI 1: Total incidentes por año
query = '''
    SELECT t.anio, COUNT(*) as total
    FROM fact_incidente f
    JOIN dim_tiempo t ON f.sk_tiempo = t.sk_tiempo
    GROUP BY t.anio ORDER BY t.anio
'''
df_anio = pd.read_sql(query, engine)

fig = px.bar(df_anio, x='anio', y='total', title='KPI 1 — Incidentes por año',
             labels={'anio': 'Año', 'total': 'Total incidentes'},
             color='total', color_continuous_scale='Blues', text='total')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# KPI 2: Distribución por gravedad
query = '''
    SELECT a.gravedad_accidente, COUNT(*) as total
    FROM fact_incidente f
    JOIN dim_accidente a ON f.sk_accidente = a.sk_accidente
    GROUP BY a.gravedad_accidente ORDER BY total DESC
'''
df_grav = pd.read_sql(query, engine)

fig = px.pie(df_grav, names='gravedad_accidente', values='total',
             title='KPI 2 — Distribución por gravedad',
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# KPI 3: Top 10 comunas
query = '''
    SELECT u.comuna, COUNT(*) as total
    FROM fact_incidente f
    JOIN dim_ubicacion u ON f.sk_ubicacion = u.sk_ubicacion
    WHERE u.comuna != 'DESCONOCIDO'
    GROUP BY u.comuna ORDER BY total DESC LIMIT 10
'''
df_com = pd.read_sql(query, engine).sort_values('total')

fig = px.bar(df_com, x='total', y='comuna', orientation='h',
             title='KPI 3 — Top 10 comunas con más incidentes',
             color='total', color_continuous_scale='Purples', text='total')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# KPI 4: Incidentes por clase de accidente
query = '''
    SELECT a.clase_accidente, COUNT(*) as total
    FROM fact_incidente f
    JOIN dim_accidente a ON f.sk_accidente = a.sk_accidente
    GROUP BY a.clase_accidente ORDER BY total DESC
'''
df_clase = pd.read_sql(query, engine)

fig = px.bar(df_clase, x='clase_accidente', y='total',
             title='KPI 4 — Incidentes por clase de accidente',
             color='total', color_continuous_scale='Teal', text='total')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Resumen final del DW
print('=' * 50)
print('RESUMEN DATA WAREHOUSE')
print('=' * 50)
for tabla in ['dim_tiempo', 'dim_ubicacion', 'dim_accidente', 'fact_incidente']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {tabla}', engine)['n'][0]
    print(f'{tabla:20s}: {count:,} filas')